
# Tahap 1: Data Auditing & Discrepancy Detection
**Dataset:** `hotel_booking_data.csv`  
**Prinsip Kualitas Data:** Completeness, Consistency, Accuracy, Uniqueness, Domain & Range Rule

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# 0. Load Dataset
url = "hf://datasets/momentarek1/hotel_booking_data/hotel_booking_data.csv"
df = pd.read_csv(url)

print("=" * 70)
print("1. RINGKASAN DIMENSI DAN TIPE DATA")
print("=" * 70)
print(f"Jumlah Baris  : {df.shape[0]:,}")
print(f"Jumlah Kolom  : {df.shape[1]}")
print("\n--- Tipe Data Kolom ---")
print(df.dtypes)

print("\n" + "=" * 70)
print("2. DETEKSI NILAI HILANG (NULL RULE - COMPLETENESS)")
print("=" * 70)
null_counts = df.isnull().sum()
null_perc = (null_counts / len(df)) * 100
missing_df = pd.DataFrame({
    'Kolom': null_counts.index,
    'Jumlah Missing': null_counts.values,
    'Persentase (%)': null_perc.round(4).values
})
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values(by='Jumlah Missing', ascending=False).reset_index(drop=True)
display(missing_df)

print("\n" + "=" * 70)
print("3. DETEKSI DUPLIKASI (UNIQUENESS RULE)")
print("=" * 70)
dup_count = df.duplicated().sum()
print(f"Jumlah baris duplikat identik: {dup_count}")

print("\n" + "=" * 70)
print("4. DETEKSI NILAI ANOMALI / MUSTAHIL (DOMAIN & RANGE RULE - ACCURACY)")
print("=" * 70)
adr_neg = df[df['adr'] < 0]
adr_ext = df[df['adr'] > 1000]
print(f"▶ Jumlah baris dengan ADR < 0 (Harga Negatif)  : {len(adr_neg)}")
if not adr_neg.empty:
    display(adr_neg[['hotel', 'adr', 'is_canceled']])

print(f"\n▶ Jumlah baris dengan ADR > 1000 (Nilai Ekstrem) : {len(adr_ext)}")
if not adr_ext.empty:
    display(adr_ext[['hotel', 'adr', 'is_canceled']])

# Deteksi Total Tamu == 0
total_guests = df['adults'] + df['children'].fillna(0) + df['babies'].fillna(0)
zero_guests = df[total_guests == 0]
print(f"\n▶ Jumlah baris di mana total tamu (adults + children + babies) == 0 : {len(zero_guests)}")

print("\n" + "=" * 70)
print("5. DETEKSI FITUR SENSITIF (PII) & POTENSI DATA LEAKAGE")
print("=" * 70)
pii_cols = ['name', 'email', 'phone-number', 'credit_card']
leakage_cols = ['reservation_status', 'reservation_status_date']

existing_pii = [col for col in pii_cols if col in df.columns]
existing_leakage = [col for col in leakage_cols if col in df.columns]

print(f"Kolom PII (Privasi) yang terdeteksi        : {existing_pii}")
print(f"Kolom Data Leakage yang terdeteksi        : {existing_leakage}")

## Ringkasan Hasil Temuan Anomali & Rekomendasi Data Cleaning

Berdasarkan hasil audit data kualitas pada `hotel_booking_data.csv`, berikut adalah rincian temuan anomali beserta dampaknya:

### 1. Completeness (Keberadaan Nilai Hilang)
- **`company`**: Terdapat **112,593 missing values (94.31%)**. Karena persentase missing sangat tinggi, kolom ini tidak informatif dan sebaiknya **dihapus (dropped)** atau diisi kategori `'Unknown'`.
- **`agent`**: Terdapat **16,340 missing values (13.69%)**. Dapat diisi dengan nilai `0` (mengindikasikan pemesanan langsung tanpa agen).
- **`country`**: Terdapat **488 missing values (0.41%)**. Dapat diisi dengan nilai modus (misal `'PRT'`) atau `'Unknown'`.
- **`children`**: Terdapat **4 missing values (0.003%)**. Dapat diimputasi dengan nilai `0`.

### 2. Uniqueness (Keunikan Baris Data)
- **Duplikat Identik**: Ditemukan **0 baris duplikat**. Hal ini dipengaruhi oleh adanya kolom PII unik (`email`, `phone-number`, `credit_card`). Jika kolom PII dihapus, duplikasi entri pemesanan perlu diperiksa kembali.

### 3. Accuracy & Domain/Range Rule (Anomali Nilai Mustahil)
- **ADR Negatif (`adr < 0`)**: Ditemukan **1 baris** dengan nilai `adr = -6.38`. Harga sewa kamar per malam tidak mungkin bernilai negatif. Baris ini perlu **dihapus** atau dikoreksi.
- **ADR Ekstrem (`adr > 1000`)**: Ditemukan **1 baris** dengan nilai `adr = 5400.0`. Nilai ini merupakan outlier ekstrem (jauh di atas rata-rata ~101.83). Baris ini perlu **dihapus** atau ditangani saat preprocessing.
- **Total Tamu Nol (`total_guests == 0`)**: Ditemukan **180 baris** di mana jumlah `adults + children + babies = 0`. Reservasi hotel tanpa tamu adalah anomali data dan baris-baris ini harus **dihapus**.

### 4. Privacy & Data Leakage Prevention
- **PII (Personally Identifiable Information)**: Kolom `name`, `email`, `phone-number`, dan `credit_card` harus **dihapus (dropped)** demi kepatuhan privasi data dan menghindari overfitting.
- **Data Leakage**: Kolom `reservation_status` dan `reservation_status_date` mencatat informasi *setelah* reservasi berakhir (`Check-Out`, `Canceled`, `No-Show`). Jika variabel target adalah `is_canceled`, menyertakan kolom ini akan menyebabkan **100% data leakage** pada model machine learning. Kolom ini harus **dihapus** sebelum pemodelan.